# ListCVP using fpylll enumeration

In [2]:
from lattices import *

In [15]:
def create_instance_listcvp(n, k, z=7, p=127, quiet=False):
    E = [2**i for i in range(z)]
    mu = QQ(sum(map(int, E)))/len(E)
    Fp = GF(p)
    global G, A
    # Initial ResSD problem
    while True:
        H = random_matrix(Fp,n-k,n)
        if H.rank() != n-k:
            continue
        C = LinearCode(H).dual_code()
        G = C.systematic_generator_matrix()
        if G[:k,:k] != 1:  # don't want to shuffle columns
            continue
        break

    assert H*G.T == 0

    # Instance
    e = vector(Fp,[choice(E) for i in range(n)])
    if not quiet:
        print("secret e", e)
    s = e*(H.T)
    y = H.solve_right(s)
    assert y*H.T == s

    # Lattice
    A = matrix(ZZ, G).stack(zero_matrix(n-k,k).augment(p*identity_matrix(n-k)))

    a = H.solve_right(s)  # base solution
    target = vector(QQ, [mu - int(ai) for ai in a])
    phie = vector(ZZ, [int(ei) - int(ai) for ei, ai in zip(e, a)])

    diff = vector(QQ, phie) - target
    dist = RR(diff.norm())
    R = RR(n**0.5 * sum((int(ri) - mu)**2 * QQ(1) / len(E) for ri in E)**0.5)
    A.solve_left(phie).change_ring(ZZ)  # check that phie in the lattice
    secret = dist, e, phie
    return A, target, R, a, mu, secret

## Create instance

In [9]:
n = 35
k = 21
z = 4
E = [2**i for i in range(z)]

A0, target, R, a, mu, secret = create_instance_listcvp(n=n, k=k, z=z)
dist, e, phie = secret
diff = vector(QQ, phie) - target
msg = A0.solve_left(phie).change_ring(ZZ)
print("dist", dist, "<?", "R", R, ":", dist < R)
if dist >= R:
    print("WARNING: enumeration will fail, resample!")

secret e (4, 2, 8, 1, 4, 8, 2, 1, 4, 1, 8, 4, 4, 4, 2, 1, 4, 1, 2, 8, 1, 1, 1, 4, 1, 2, 1, 2, 4, 4, 4, 1, 4, 2, 4)
dist 13.3299474867683 <? R 15.8607219255619 : True


## ListCVP using enumeration

In [10]:
A = IntegerMatrix.from_matrix(A0)
A.ncols

35

In [11]:
FPLLL.set_random_seed(2025)
_ = FPLLL.set_threads(1)

A = IntegerMatrix.from_matrix(A0)
A = BKZ.reduction(A, BKZ.Param(n))  # BKZ-n
Asage = matrix(ZZ, A.to_matrix([[0] * n for _ in range(n)]))

M = GSO.Mat(A)
M.update_gso()

BOUND = R**2
print("BOUND/mu_0,0:", RR(R**2 / M.get_r(0,0)))

enum = Enumeration(M, 2**30, strategy=EvaluatorStrategy.FIRST_N_SOLUTIONS, sub_solutions=0)#, callbackf=test)

tf = M.from_canonical(map(float, target))
out = enum.enumerate(0, Asage.nrows(), BOUND, 0, target=tf, pruning=None)#pruning.coefficients)
#out = enum.enumerate(0, Asage.nrows(), BOUND, 0, pruning=None)#pruning.coefficients)

out2 = []
print("got vectors", len(out), "= 2^%.2f" % LOG2(len(out)))
for norm, x in out[:10]:
    lattice_vector = A.multiply_left(x)
    print(
        "%.1f" % norm, lattice_vector[:10],
    )
print("...")

BOUND/mu_0,0: 2.11397058823529
got vectors 1199372 = 2^20.19
108.7 (-21.0, -93.0, -81.0, -26.0, -106.0, -13.0, -22.0, -82.0, -45.0, -56.0)
117.2 (-24.0, -99.0, -79.0, -23.0, -104.0, -13.0, -26.0, -82.0, -44.0, -58.0)
119.2 (-24.0, -90.0, -81.0, -24.0, -106.0, -12.0, -27.0, -84.0, -44.0, -56.0)
119.7 (-23.0, -93.0, -78.0, -23.0, -105.0, -12.0, -29.0, -79.0, -47.0, -58.0)
119.7 (-24.0, -93.0, -81.0, -25.0, -102.0, -12.0, -22.0, -82.0, -45.0, -57.0)
120.7 (-24.0, -92.0, -78.0, -22.0, -107.0, -11.0, -26.0, -81.0, -49.0, -54.0)
122.7 (-23.0, -94.0, -80.0, -24.0, -106.0, -12.0, -25.0, -82.0, -41.0, -57.0)
125.2 (-22.0, -92.0, -80.0, -24.0, -105.0, -13.0, -27.0, -84.0, -48.0, -55.0)
126.7 (-22.0, -94.0, -82.0, -24.0, -106.0, -12.0, -26.0, -84.0, -47.0, -58.0)
127.2 (-22.0, -93.0, -82.0, -25.0, -105.0, -11.0, -27.0, -80.0, -47.0, -60.0)
...


## Check output for solutions

In [12]:
cols = Asage.columns()
for norm, x in tqdm(out):    
    x = vector(ZZ, x)
    sol = []
    for i, col in enumerate(cols):
        sol.append(x * col + int(a[i]))
        if sol[-1] not in E:
            break
    else:
        print("solution", sol)

  1%|█▉                                                                                                                                                                                            | 12450/1199372 [00:00<00:19, 62449.70it/s]

solution [4, 2, 8, 1, 4, 8, 2, 1, 4, 1, 8, 4, 4, 4, 2, 1, 4, 1, 2, 8, 1, 1, 1, 4, 1, 2, 1, 2, 4, 4, 4, 1, 4, 2, 4]


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1199372/1199372 [00:18<00:00, 63698.20it/s]


# Some statistics

## Median heuristic success rate

In [17]:
cnt = total = 0
for _ in tqdm(range(10**5)):
    total += 1
    A0, target, R, a, mu, secret = create_instance_listcvp(n=n, k=k, z=z, quiet=True)
    dist, e, phie = secret
    cnt += dist <= R
cnt / total

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100000/100000 [03:12<00:00, 520.24it/s]


0.51009

## Exact (non-GSA) complexity of enumeration

Log2 values:

In [18]:
costs = []
for _ in tqdm(range(10**3)):
    A0, target, R, a, mu, secret = create_instance_listcvp(n=n, k=k, z=z, quiet=True)
    A = IntegerMatrix.from_matrix(A0)
    A = BKZ.reduction(A, BKZ.Param(n))
    cost = enum_cost(A, R, pruning=False)
    costs.append(cost)
print("min", RR(log(min(costs), 2)))
print("avg", RR(log(sum(costs)/len(costs), 2)))
print("med", RR(log(costs[len(costs)//2], 2)))
print("max", RR(log(max(costs), 2)))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:26<00:00, 37.45it/s]

min 27.9199405313248
avg 28.4522928737564
med 28.7504522207742
max 28.8395470548722


Versus GSA-based:

In [26]:
vol = 127**(n-k-1)
delta0_hkz = RR(ball_vol(n)**(-1/n**2))
norm = R
LOG2(enum_cost_delta(n, vol, norm, delta0_hkz, pruning=False))

32.8905557637630

Pruning is not useful at this dimension:

In [28]:
LOG2(enum_cost_delta(n, vol, norm, delta0_hkz, pruning=True))

34.5703166285849